In [1]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [2]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [3]:
# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating \
this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
1.Y/N indicating whether the article is talking about a region of Boston. \n 2.The specific location within the city you got if you got Y in the first question. \
3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. PLEASE CONSIDER THE CONTEXT OF THE ARTICLE. Give your response in the following format: \
# 1. A very brief summary of what the article is talking about. \n 2.The specific location you chose based on the context of the article. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. \n\n
# Headline: \n\n {headline} \n\n [/INST]""",
# )

# prompt = PromptTemplate(
#     input_variables=["headline"],
#     template="""<<SYS>> \n You are an assistant tasked in geo-locating \
# this news article. \n <</SYS>> \n\n [INST] Generate a SHORT response \
# of where you think this article is talking about. BE SPECIFIC AND CONCISE AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
# 1.Y/N indicating whether the article is talking about a region of Boston \n 2.The specific location within the city you got if you got Y in the first question. \
# If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. PLEASE KEEP YOUR ANSWER SHORT. \n\n
# Headline: \n\n {headline} \n\n Body: \n\n {body}  \n\n [/INST]""",
# )

In [4]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"

In [5]:
llm = LlamaCpp(
    model_path=llama_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
    callback_manager=CallbackManager([StreamingStdOutCallbackHandler()]),
    verbose=True,
)
output_parser = StrOutputParser()

llama_model_loader: loaded meta data with 19 key-value pairs and 291 tensors from ./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf (version GGUF V2)
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = llama
llama_model_loader: - kv   1:                               general.name str              = LLaMA v2
llama_model_loader: - kv   2:                       llama.context_length u32              = 4096
llama_model_loader: - kv   3:                     llama.embedding_length u32              = 4096
llama_model_loader: - kv   4:                          llama.block_count u32              = 32
llama_model_loader: - kv   5:                  llama.feed_forward_length u32              = 11008
llama_model_loader: - kv   6:                 llama.rope.dimension_count u32              = 128
llama_model_loader: - kv   7:                 llama.attention.head_count u

In [6]:
chain = prompt | llm | output_parser

In [7]:
# Run LLM on a given article
def run_llm(headline, body):
    return chain.invoke({"headline": headline, "body": body})

## NER Model

In [8]:
import spacy
from span_marker import SpanMarkerModel

In [9]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [10]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [11]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [12]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [13]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [14]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file)

## Pipeline Entry Point

In [15]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one
# sample_data_dir = "./sample_data/se_naacp_db.articles_data.csv"

In [16]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 10 articles
raw_df = full_df.sample(50)
# raw_df = full_df
len(raw_df)


50

In [17]:
# raw_df = pd.read_csv(sample_data_path)

In [18]:
raw_df.head(10)

,_id,Type,Label,hl1,Byline,Section Navigation,Section,Title,Paths,Publish Date,Has Path?,body
159,00000175-9a93-d944-a9fd-dad362f00001,Article,Where The Whirlwind Of Trump Election Lawsuits...,Where The Whirlwind Of Trump Election Lawsuits...,Greater Boston Staff,NaN,National News,NaN,/national-news/2020/11/05/where-the-whirlwind-...,Thu Nov 05 19:41:21 EST 2020,TRUE,President Donald Trump and the Republicans hav...
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,Article,"'The Very, Very Tip Of A Huge Iceberg': Why Ha...","'The Very, Very Tip Of A Huge Iceberg': Why Ha...","Marilyn Schairer, Meghan Smith",NaN,Local News,NaN,/local-news/2021/08/27/the-very-very-tip-of-a-...,Fri Aug 27 06:00:28 EDT 2021,TRUE,"From an attack on <a href=""https://www.wgbh.or..."
12319,00000186-7a1d-d717-adce-fa1d2d250001,Article,Dave Epstein Forecast: Expect a mixed bag of p...,Dave Epstein Forecast: Expect a mixed bag of p...,Dave Epstein,NaN,Local News,NaN,/local-news/2023/02/22/dave-epstein-forecast-e...,Wed Feb 22 13:42:48 EST 2023,TRUE,You likely noticed the clouds have been increa...
10141,00000183-1324-d40f-a98b-1bbf592d0001,Article,"Eric In The Evening: Saturday ,September 3, 2022","Eric In The Evening: Saturday ,September 3, 2022",00000183-1324-d40f-a98b-1bbf592d0000,NaN,Jazz,NaN,/jazz/2022/09/06/eric-in-the-evening-saturday-...,Tue Sep 06 10:56:29 EDT 2022,TRUE,00000183-1324-d40f-a98b-1bbf592d0002
2118,00000177-7cca-d244-a57f-7ffbd69c0001,Article,"For Her 'SNL' Debut, Phoebe Bridgers Goes Bigg...","For Her 'SNL' Debut, Phoebe Bridgers Goes Bigg...",Stephen Thompson,NaN,Arts & Culture,NaN,/arts-culture/2021/02/08/for-her-snl-debut-pho...,Sun Feb 07 08:06:00 EST 2021,TRUE,Most of us spent 2020 in a holding pattern — i...
7472,0000017e-c560-d578-a77e-c571b1550001,Article,"With Jeff Zucker out, the future of CNN is unc...","With Jeff Zucker out, the future of CNN is unc...",Greater Boston Staff,NaN,News,NaN,/news/2022/02/04/with-jeff-zucker-out-the-futu...,Fri Feb 04 13:07:38 EST 2022,TRUE,CNN President Jeff Zucker stepped down this we...
966,00000176-62bd-d4fd-a17e-e7bd67960001,Article,'Pride & Prejudice' Episode 6 Recap: Rumor Has It,'Pride & Prejudice' Episode 6 Recap: Rumor Has It,Jackie Bruleigh,NaN,Programs,NaN,/programs/2021/01/11/pride-prejudice-episode-6...,Mon Jan 11 09:00:14 EST 2021,TRUE,"<i>Every season, the </i>Drama After Dark<i> t..."
9770,00000182-5053-d463-abf3-765f71df0001,Article,There's a familiar ring to the latest Dershowi...,There's a familiar ring to the latest Dershowi...,Callie Crossley,NaN,Commentary,NaN,/commentary/2022/08/01/theres-a-familiar-ring-...,Mon Aug 01 05:00:58 EDT 2022,TRUE,It’s deja vu all over again on Martha’s Vineya...
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Article,"Wednesday, January 26","Wednesday, January 26",0000017e-919d-d1c4-a7fe-bb9f30a30000,NaN,Digital Mural,NaN,/digital-mural/2022/01/26/wednesday-january-26...,Wed Jan 26 00:00:47 EST 2022,TRUE,Celebrate <i>ZOOM</i>’s 50th anniversary! Join...
11716,00000185-c5d2-d12f-a1df-cfded1df0001,Article,"The world's oldest person, Sister André of Fra...","The world's oldest person, Sister André of Fra...",Kaitlyn Radde,NaN,International News,NaN,/international-news/2023/01/19/the-worlds-olde...,Wed Jan 18 11:04:00 EST 2023,TRUE,"Sister André, the world's oldest known person,..."


The ML Model honestly just needs the `id`, `header`, and `body`.

In [19]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

In [20]:
# For Testing Purposes Only
# df = df[:20]

In [21]:
df["llama_prediction"] = None # Add the llama_prediction

Remove Duplicates (if any)

In [22]:
duplicates = df.duplicated(subset=['hl1'])

In [23]:
print(duplicates.value_counts())

False    50
Name: count, dtype: int64


In [24]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [25]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

  0%|          | 0/50 [00:00<?, ?it/s]C:\Users\axel0\AppData\Local\Temp\ipykernel_14584\1969457438.py:2: MarkupResemblesLocatorWarning: The input looks more like a filename than markup. You may want to open this file and pass the filehandle into Beautiful Soup.
  func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
100%|██████████| 50/50 [00:00<00:00, 3399.06it/s]


Clean the Body and Header with Regex

In [26]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

100%|██████████| 50/50 [00:00<?, ?it/s]


### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary

In [27]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

In [28]:
# TODO: check for unwanted locations
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            return location  
    return None

In [29]:
df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)

100%|██████████| 50/50 [00:00<00:00, 5258.26it/s]


In [30]:
df["Explicit_Pass"].value_counts().head(10)

Explicit_Pass
Boston Public Radio    3
New                    2
BU                     1
Boston City Council    1
Name: count, dtype: int64

### NER Code First Pass

In [31]:
unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [32]:
# Return the first valid facility found, or organization if none are found
def valid_facility(entities, LLMRun):
    first_org = None
    valid_org = False
    first_fac = None

    print(entities)

    for entity in entities:
        # If it's a valid facility, return it
        if (entity.label_ == "FAC"):
            if (first_fac == None): # Save first facility
                first_fac = entity.text
                print("facility", entity.text)
            if (entity.text not in unwanted_entities["FAC"]): # If facility is valid, return it
                print("valid facility", entity.text)

                return entity.text
        
        # Check for organizations in case no facilities are found
        elif (entity.label_ == "ORG"):
            if (first_org == None): # Save first organization
                first_org = entity.text
                print("org", entity.text)
                if (entity.text not in unwanted_entities["ORG"]): # Check if it's valid
                    valid_org = True
                    print("valid org", entity.text)
            elif ( (not valid_org) and entity.text not in unwanted_entities["ORG"]): # Only switch it for a valid organization
                first_org = entity.text
                valid_org = True
                print("valid org", entity.text)

    # For LLM Prediciton, can accept an organization too
    if (LLMRun):
        if (first_org != None):
            return first_org
    return None

In [33]:
# Run NER on the body of the article and return first valid facility
def predict_NER_def(text, LLMRun=False):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        return valid_facility(entities, LLMRun)
        
    except Exception as error:
        return None

In [34]:
# Run NER on the articles that do not have an explicit location in the title
def explicit_filtering_NER(article):
    try:
        # If the article does not have an explicit location, run NER
        if (article['Explicit_Pass'] != None): 
            print(f"Has location from title: {article['hl1']}")
            return None
        else:
            return predict_NER_def(article['body'])
    except Exception as error:
        print(error)
        return None

In [35]:
df['NER_Pass'] = df.progress_apply(explicit_filtering_NER, axis=1)

  4%|▍         | 2/50 [00:11<04:36,  5.77s/it]

(Donald Trump, Republicans, two, Georgia, Michigan, Thursday, afternoon, Jim Braude, Margery Eagan, GBH News, Jeannie Suk Gersen, Harvard Law School, New Yorker)
org GBH News
valid org GBH News


  6%|▌         | 3/50 [01:44<32:52, 41.97s/it]

(Asian American, Atlanta, Brighton, two, Black, Winthrop, this year, the Institute on Race and Justice, Northeastern University, Carlos Cuevas, the Violence and Justice Research Lab, Northeastern University, Marilyn Schairer, Morning Edition, Friday, Cuevas, First, Cuevas, Latino, only 8%, Cuevas, ModuleDaniel Medwed, GBH News, Northeastern University, Morning Edition, first, Second, Medwed, third, Cuevas, Schairer, George Floyd, May 2020, Cuevas, Cuevas)
org the Institute on Race and Justice
valid org the Institute on Race and Justice


  8%|▊         | 4/50 [03:21<47:53, 62.48s/it]

...

(AP, Eastern Conference, Boston Celtics, Ime Udoka, months, Wyc Grousbeck, Friday, Grousbeck, day, Eastern Conference, Udoka, 2022 23, one, Grousbeck, Brad Stevens, two days ago, The Associated Press, only one, Celtics, Friday, Grousbeck, Udoka, Grousbeck, first, three months, the NBA Finals, Udoka, days, Celtics, this season, Joe Mazzulla, June 30 2023, Udoka, Stevens, yesterday, Stevens, Grousbeck, earlier this summer, Grousbeck, Udoka, Grousbeck, year, Udoka, Celtics, Paul Pierce, Grousbeck, Udoka, Mazzulla 34, West Virginia, 2007, NIT, ninth, Duke, NCAA, two year, 2017 19, Fairmont State, West Virginia, Joe, Stevens, Tuesday, Tim Reynolds, Miami)
org AP
valid org AP


100%|██████████| 50/50 [1:31:28<00:00, 161.28s/it]

(the holiday season, 2021, Religion News Service, 13 to 25, Revs, Irene Monroe, Emmett G. Price III, Boston Public Radio, the United States, Price, Gen, around 11, Price, Monroe, Monroe, Price, Monroe, Monroe, Boston, Detour African American Heritage Trail, All Rev Up, Price, Community of Love Christian Fellowship, Allston, Africana, Berklee College of Music, All Rev Up)
org Religion News Service
valid org Religion News Service


100%|██████████| 50/50 [1:31:32<00:00, 109.86s/it]

()


In [36]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None


### Llama Prediction

In [37]:
#TODO: Comply with token limit of 2048 for Llama
# Run the LLM model on the articles that haven't been tagged with a location yet. Then run NER on the LLM prediction
def predict_llama(article):
    try:
        # If the article does not have an explicit location or NER location, run LLM
        if (article['Explicit_Pass'] != None or article['NER_Pass'] != None):
            print(f"Has location from title or NER: {article['hl1']}")
            return None
        else:
            llama_prediction = run_llm(article['hl1'], article['body'])
            print(llama_prediction)
            return predict_NER_def(llama_prediction, True)
    except Exception as error:
        print(error)
        return None

In [38]:
df['NER_Prediction'] = df.progress_apply(predict_llama, axis=1)

  0%|          | 0/50 [00:00<?, ?it/s]

  Here is my response based on the information provided in the article:
1. Y - The article is talking about a region of Boston.
2. Based on the article, I believe the specific location within Boston is likely the Federal Courthouse in Boston, where judges have rejected President Trump and the Republicans' claims of election fraud in Georgia and Michigan.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Federal Courthouse in Boston
* Georgia
* Michigan

Based on the language used in the article, it appears that the Federal Courthouse in Boston is where the lawsuits filed by President Trump and the Republicans are being heard and judged. The mention of "judges in two states" and "rejected their claims" suggests that the Federal Courthouse in Boston is the location where these rejections occurred. Additionally, the inclusion of Georgia and Michigan as specific locations in the article implies that these are the stat


llama_print_timings:        load time =   31048.15 ms
llama_print_timings:      sample time =      78.85 ms /   226 runs   (    0.35 ms per token,  2866.24 tokens per second)
llama_print_timings: prompt eval time =   31046.61 ms /   311 tokens (   99.83 ms per token,    10.02 tokens per second)
llama_print_timings:        eval time =   39244.69 ms /   225 runs   (  174.42 ms per token,     5.73 tokens per second)
llama_print_timings:       total time =   70901.45 ms /   536 tokens


  Here is my response based on the information provided in the article:
1. Y - The article is talking about a region of Boston.
2. Based on the article, I believe the specific location within Boston is likely the Federal Courthouse in Boston, where judges have rejected President Trump and the Republicans' claims of election fraud in Georgia and Michigan.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* Federal Courthouse in Boston
* Georgia
* Michigan

Based on the language used in the article, it appears that the Federal Courthouse in Boston is where the lawsuits filed by President Trump and the Republicans are being heard and judged. The mention of "judges in two states" and "rejected their claims" suggests that the Federal Courthouse in Boston is the location where these rejections occurred. Additionally, the inclusion of Georgia and Michigan as specific locations in the article implies that these are the stat

  4%|▍         | 2/50 [01:53<45:35, 56.99s/it]

...

  Based on the information provided in the article, I would estimate that it is talking about a specific location within Boston, Massachusetts. Here are my responses to your questions:
1. Y - The article is indeed talking about a region of Boston.
2. The specific location within Boston that I believe the article is referring to is the Fenway neighborhood. This is based on the mention of "Fenway" in the article's title and body.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* GBH Music (a radio station located in Boston)
* Jazz on 89.7 (a show aired on GBH Music)

Based on these details, I believe the article is referring to the Fenway neighborhood in Boston where GBH Music's Jazz on 89.7 show is broadcast from.


llama_print_timings:        load time =   31048.15 ms
llama_print_timings:      sample time =      60.63 ms /   185 runs   (    0.33 ms per token,  3051.35 tokens per second)
llama_print_timings: prompt eval time =    5615.84 ms /    67 tokens (   83.82 ms per token,    11.93 tokens per second)
llama_print_timings:        eval time =   32072.65 ms /   184 runs   (  174.31 ms per token,     5.74 tokens per second)
llama_print_timings:       total time =   38152.07 ms /   251 tokens


  Based on the information provided in the article, I would estimate that it is talking about a specific location within Boston, Massachusetts. Here are my responses to your questions:
1. Y - The article is indeed talking about a region of Boston.
2. The specific location within Boston that I believe the article is referring to is the Fenway neighborhood. This is based on the mention of "Fenway" in the article's title and body.
3. The involved specific locations or organizations explicitly found within the article that influenced my decision are:
* GBH Music (a radio station located in Boston)
* Jazz on 89.7 (a show aired on GBH Music)

Based on these details, I believe the article is referring to the Fenway neighborhood in Boston where GBH Music's Jazz on 89.7 show is broadcast from.


100%|██████████| 50/50 [1:17:53<00:00, 93.48s/it] 

(Boston, Massachusetts, 1, Boston, 2, Boston, Fenway, Fenway, 3, GBH Music, Boston, Jazz on 89.7, GBH Music, Fenway, Boston, GBH Music's, Jazz on 89.7)
facility Fenway
valid facility Fenway


In [39]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Prediction
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None,the Federal Courthouse
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None,Institute on Race and Justice
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128,None
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None,Old North Church
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None,None
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None,CNN
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants,None
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None,the Chilmark Free Library
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None,the Boylston Theatre
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None,Guinness World Records


In [40]:
## TODO: DELETE AFTER POPULATING THE UNWANTED ENTITIES CACHE
# unwanted_entities = {
#     'FAC': ['Boston'],
#     'ORG': ['New York Times'],
#     'LOC': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
#     'GPE': ['Boston', 'Massachusetts', 'Brookline', 'Allston'],
# }

# save_cache_to_file(unwanted_entities, unwanted_entities_path)

Extract locations from the most specific pass

In [41]:
# Get the locations from the most specific pass for a given article
def extractLocations(article):
    for key in ['Explicit_Pass', 'NER_Pass', 'NER_Prediction']:
        location = article.get(key)
        if location is not None:
            return location
    return None

In [42]:
df['Locations'] = df.progress_apply(extractLocations, axis=1)

100%|██████████| 50/50 [00:00<?, ?it/s]


In [43]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Prediction,Locations
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None,the Federal Courthouse,the Federal Courthouse
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None,Institute on Race and Justice,Institute on Race and Justice
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128,None,Route 128
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None,Old North Church,Old North Church
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None,None,BU
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None,CNN,CNN
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants,None,JudgyPants
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None,the Chilmark Free Library,the Chilmark Free Library
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None,the Boylston Theatre,the Boylston Theatre
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None,Guinness World Records,Guinness World Records


In [45]:
df

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Prediction,Locations
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None,the Federal Courthouse,the Federal Courthouse
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None,Institute on Race and Justice,Institute on Race and Justice
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128,None,Route 128
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None,Old North Church,Old North Church
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None,None,BU
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None,CNN,CNN
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants,None,JudgyPants
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None,the Chilmark Free Library,the Chilmark Free Library
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None,the Boylston Theatre,the Boylston Theatre
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None,Guinness World Records,Guinness World Records


## Get the coordinates

In [47]:
known_locations_path = "./geodata/known_locations.json"  
known_locations = load_cache(known_locations_path)

In [48]:
# Get the coordinates of the location
def getCoordinates(location): # Valid labels are FAC for NER_Pass; FAC and ORG for NER_Prediction
    if (location == None or len(location) == 0): return None  
    
    # Only get coordinates if the location is not already known
    if (location in known_locations):
        longitude, latitude = known_locations[location]["coordinates"]
    else:
        # Get coordinates and save to cache
        longitude, latitude = callGoogleMapsAPI(location)
        known_locations[location] = {"coordinates": [longitude, latitude], "tract": None, "county": None}
        save_cache_to_file(known_locations, known_locations_path)

    return [longitude, latitude]

In [49]:
df['Coordinates'] = df['Locations'].progress_apply(getCoordinates)

100%|██████████| 50/50 [00:05<00:00,  9.76it/s]


In [50]:
df.head(10)

,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None,the Federal Courthouse,the Federal Courthouse,"[-71.3824374, 42.4072107]"
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None,Institute on Race and Justice,Institute on Race and Justice,"[-71.3824374, 42.4072107]"
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128,None,Route 128,"[-70.9990196, 42.5166695]"
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None,Old North Church,Old North Church,"[-71.05439439999999, 42.3663259]"
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None,None,BU,"[-71.1053991, 42.3504997]"
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None,CNN,CNN,"[-71.3824374, 42.4072107]"
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants,None,JudgyPants,"[-71.3824374, 42.4072107]"
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None,the Chilmark Free Library,the Chilmark Free Library,"[-70.74475009999999, 41.3431688]"
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None,the Boylston Theatre,the Boylston Theatre,"[-71.0642623, 42.3518662]"
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None,Guinness World Records,Guinness World Records,"[-71.3824374, 42.4072107]"


## Geocode locations

In [51]:
# Get the census tract of the location
def query_census_api(location, coordinates):
    longitude, latitude = coordinates
    base_url = f'https://geocoding.geo.census.gov/geocoder/geographies/coordinates?'
    survey_ver = f'&benchmark=4&vintage=4&layers=2020 Census Blocks&format=json'
    url = f'{base_url}x={longitude}&y={latitude}{survey_ver}'

    response = requests.get(url)

    # Check if response is valid
    if (response.status_code == 200):
        results = response.json()
        try:
            tract = results['result']['geographies']['2020 Census Blocks'][0]['TRACT']
            county = results['result']['geographies']['2020 Census Blocks'][0]['COUNTY']

            return tract, county
        except IndexError:
            print("Unable to retrieve census geography for: " + location)
        except KeyError:
            print("Location is outside of the United States: " + location)
        except Exception as error:
            print(error)

    return None, None  # Return this if API call failed or no tracts found

In [68]:
# Get the census tract and county of the location
def geocode(location):
    if (location is None or len(location) == 0): return None, None  

    # Only geocode if it's not known
    Tract = known_locations[location]["tract"],
    County = known_locations[location]["county"]
    if (Tract is None or County is None):
        # Geocode article
        coordinates = known_locations[location]["coordinates"]
        Tract, County = query_census_api(location, coordinates)

        # Save to cache
        known_locations[location]["tract"] = Tract
        known_locations[location]["county"] = County
        save_cache_to_file(known_locations, known_locations_path)
    
    return Tract, County
    

In [70]:
df[['Tracts', 'County']] = df.progress_apply(lambda row: pd.Series(geocode(row['Locations'])), axis=1)
df

100%|██████████| 50/50 [00:17<00:00,  2.80it/s]


,_id,hl1,body,llama_prediction,Explicit_Pass,NER_Pass,NER_Prediction,Locations,Coordinates,Tracts,County
159,00000175-9a93-d944-a9fd-dad362f00001,Where The Whirlwind Of Trump Election Lawsuits...,President Donald Trump and the Republicans hav...,None,None,None,the Federal Courthouse,the Federal Courthouse,"[-71.3824374, 42.4072107]",365100,017
5400,0000017b-83e9-d5a5-a37f-83ef662a0001,The Very Very Tip Of Huge Iceberg Why Hate Cri...,From an attack on Asian American women in Atla...,None,None,None,Institute on Race and Justice,Institute on Race and Justice,"[-71.3824374, 42.4072107]",365100,017
12319,00000186-7a1d-d717-adce-fa1d2d250001,Dave Epstein Forecast Expect mixed bag of prec...,You likely noticed the clouds have been increa...,None,None,Route 128,None,Route 128,"[-70.9990196, 42.5166695]",209100,009
10141,00000183-1324-d40f-a98b-1bbf592d0001,Eric In The Evening Saturday September 2022,00000183 1324 d40f a98b 1bbf592d0002,None,None,None,Old North Church,Old North Church,"[-71.05439439999999, 42.3663259]",030400,025
2118,00000177-7cca-d244-a57f-7ffbd69c0001,For Her SNL Debut Phoebe Bridgers Goes Bigger ...,Most of us spent 2020 in holding pattern if we...,None,BU,None,None,BU,"[-71.1053991, 42.3504997]",010103,025
7472,0000017e-c560-d578-a77e-c571b1550001,With Jeff Zucker out the future of CNN is unce...,CNN President Jeff Zucker stepped down this we...,None,None,None,CNN,CNN,"[-71.3824374, 42.4072107]",365100,017
966,00000176-62bd-d4fd-a17e-e7bd67960001,Pride Prejudice Episode Recap Rumor Has It,Every season the Drama After Dark team gathers...,None,None,JudgyPants,None,JudgyPants,"[-71.3824374, 42.4072107]",365100,017
9770,00000182-5053-d463-abf3-765f71df0001,There familiar ring to the latest Dershowitz c...,It deja vu all over again on Martha Vineyard w...,None,None,None,the Chilmark Free Library,the Chilmark Free Library,"[-70.74475009999999, 41.3431688]",200400,007
7307,0000017e-919d-d1c4-a7fe-bb9f30a30001,Wednesday January 26,Celebrate ZOOM 50th anniversary! Join David Ka...,None,None,None,the Boylston Theatre,the Boylston Theatre,"[-71.0642623, 42.3518662]",070202,025
11716,00000185-c5d2-d12f-a1df-cfded1df0001,The world oldest person Sister Andr of France ...,Sister Andr the world oldest known person died...,None,None,None,Guinness World Records,Guinness World Records,"[-71.3824374, 42.4072107]",365100,017


In [74]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

7


Explicit_Pass
Boston Public Radio    3
New                    2
BU                     1
Boston City Council    1
Name: count, dtype: int64

In [76]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

8


NER_Pass
Route 128                          1
JudgyPants                         1
Capitol                            1
the National Lynching Memorial     1
North Shore N.E. Aquarium          1
The Triangle Shirtwaist Factory    1
Mar Lago                           1
Applebee                           1
Name: count, dtype: int64

In [77]:
print(df['NER_Prediction'].value_counts().sum())
df['NER_Prediction'].value_counts()

32


NER_Prediction
the Federal Courthouse                            1
Institute on Race and Justice                     1
Community of Love Christian Fellowship            1
Celtics                                           1
African American Heritage Trail                   1
the Fenway Community Health Center                1
Amazon                                            1
Biogen                                            1
The University of California San Diego            1
Bountiful Farms                                   1
INDER                                             1
Paycheck Protection Program                       1
Philadelphia International Records                1
FBI                                               1
Frontline                                         1
HUD                                               1
Boston Election Department                        1
Boston Public Radio                               1
Chelsea Monroe Cassel                            

In [ ]:
df

In [ ]:
df.head(10)

In [ ]:
len(df)

In [ ]:
df = df.dropna(subset=["Tracts", "County"]) # Clean those that don't have a Tract or a County

In [ ]:
print(len(df))
df.head(10)

## Topic Modeling

In [ ]:
import os
import tiktoken
import numpy as np
from transformers import pipeline
from sklearn.metrics import adjusted_rand_score
from openai import OpenAI, AsyncOpenAI
from sklearn.metrics.pairwise import cosine_similarity
from tenacity import retry, wait_random_exponential, stop_after_attempt

## OpenAI Client

In [ ]:
# Retry up to 10 times with exponential backoff, starting at 1 second and maxing out at 20 seconds delay
@retry(wait=wait_random_exponential(min=1, max=20), stop=stop_after_attempt(10))
def get_embedding(text: str, model="text-embedding-3-small"):
    #print(text)
    try:
        embedding = client.embeddings.create(input=text, model=model).data[0].embedding
        return embedding
    except Exception as e:
        print(f"Failed to retrieve ADA Embedding: {e}. Replacing with replacement value!")
        return [-1.0]
    return 

In [ ]:
client = OpenAI(
    api_key='YOUR_KEY_HERE',
)

## Taxonomy Lists

Content Taxanomy

In [ ]:
# Get the embedding for taxonomy
taxonomy_df = pd.read_csv('./taxonomy_list/Content_Taxonomy.csv', skiprows=5, usecols=range(8))
taxonomy_df.columns = taxonomy_df.iloc[0]
taxonomy_df = taxonomy_df.tail(-1)

tier_1_list = []
tier_2_list = []
tier_3_list = []
tier_4_list = []
for index, row in taxonomy_df.iterrows():
    if not pd.isnull(row['Tier 4']) and row['Tier 4'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_4_label = row['Tier 4']
        tier_4_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label} - {tier_4_label}')
    elif not pd.isnull(row['Tier 3']) and row['Tier 3'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_3_label = row['Tier 3']
        tier_3_list.append(f'{tier_1_label} - {tier_2_label} - {tier_3_label}')
    elif not pd.isnull(row['Tier 2']) and row['Tier 2'] != ' ':
        tier_1_label = row['Tier 1']
        tier_2_label = row['Tier 2']
        tier_2_list.append(f'{tier_1_label} - {tier_2_label}')
    else:
        tier_1_label = row['Tier 1']
        tier_1_list.append(f'{tier_1_label}')

tier_1_list = list(set(tier_1_list))
tier_2_list = list(set(tier_2_list))
tier_3_list = list(set(tier_3_list))
tier_4_list = list(set(tier_4_list))

tier_1_embedding = [get_embedding(topic) for topic in tier_1_list]
tier_2_embedding = [get_embedding(topic) for topic in tier_2_list]
tier_3_embedding = [get_embedding(topic) for topic in tier_3_list]
tier_4_embedding = [get_embedding(topic) for topic in tier_4_list]

all_topics_list = []
[all_topics_list.append(topic) for topic in tier_1_list]
[all_topics_list.append(topic) for topic in tier_2_list]
[all_topics_list.append(topic) for topic in tier_3_list]
[all_topics_list.append(topic) for topic in tier_4_list]

all_topics_embedding = []
[all_topics_embedding.append(embedding) for embedding in tier_1_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_2_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_3_embedding]
[all_topics_embedding.append(embedding) for embedding in tier_4_embedding]
print(len(all_topics_embedding))

Selected Taxonomy List

In [ ]:
# Get embedding for the 230 topics selected by BERTopic 
selected_taxonomy_df = pd.read_csv('./topics/embedding_similarity_label.csv')
selected_taxonomy_df = selected_taxonomy_df.dropna(subset=['closest_topic'])
selected_topics_list = selected_taxonomy_df['closest_topic'].values.tolist()

selected_topics_embedding = [get_embedding(topic) for topic in selected_topics_list]

Client Taxonomy List

In [ ]:
# Alternative taxonomy: client's list of topics
client_taxonomy_df = pd.read_excel('./topics/Asad_Topics_List.xlsx', names=['label'])
client_taxonomy_df['ada_embedding'] = client_taxonomy_df['label'].map(get_embedding)

## Obtaining Ada Embedding

In [ ]:
def truncate(tokens, length=500):
    """
    Function to get the first 500 elements from a list
    """
    return tokens[:length]

In [ ]:
df['topic_model_body'] = df['body'].apply(lambda x: re.sub(re.compile('<.*?>'), '', x))
df['tokens'] = df['topic_model_body'].apply(lambda x: x.split())
df['tokens'] = df['tokens'].apply(truncate)

In [ ]:
df['ada_embedding'] = df.tokens.apply(lambda x: get_embedding(','.join(map(str,x)), model='text-embedding-3-small'))

## Similarity Matching After Ada Embedding

In [ ]:
# Find most similar taxonomy (out of all toipcs) to news body
closest_topic_list_all = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in all_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = all_topics_list[closest_topic_index]
    closest_topic_list_all.append(closest_topic)

df['closest_topic_all'] = closest_topic_list_all

In [ ]:
# Find most similar taxonomy (out of 230 selected topics) to news body
closest_topic_list_selected = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in selected_topics_embedding]

    # Find the index of the topic with the highest similarity
    closest_topic_index = np.argmax(similarities)

    # Retrieve the closest topic embedding
    closest_topic = selected_topics_list[closest_topic_index]
    closest_topic_list_selected.append(closest_topic)

df['closest_topic_selected'] = closest_topic_list_selected

In [ ]:
client_topic_embedding_list = client_taxonomy_df['ada_embedding'].to_list()
client_topic_list = client_taxonomy_df['label'].to_list()
similarity_arr = []

closest_topic_list_client = []
for index, row in df.iterrows():
    target_embedding = row['ada_embedding']
    similarities = [cosine_similarity(np.array(target_embedding).reshape(1, -1), np.array(topic).reshape(1, -1))[0][0] for topic in client_topic_embedding_list]
    
    if max(similarities) > 0.25:    
        closest_topic_index = np.argmax(similarities) # Find the index of the topic with the highest similarity
        closest_topic = client_topic_list[closest_topic_index] # Retrieve the closest topic embedding
        closest_topic_list_client.append(closest_topic)
    else:
        closest_topic_list_client.append('Other')
    similarity_arr.append(max(similarities))
    
df['closest_topic_client'] = closest_topic_list_client

In [ ]:
df

In [ ]:
df.to_csv("./outputs/gbh_output.csv")

In [ ]:
raw_df

In [ ]:
df

In [ ]:
merged_df = pd.merge(raw_df, df, on='_id', how='inner')

In [ ]:
merged_df

In [ ]:
merged_df.to_csv("./outputs/gbh_output_all_fields.csv")

In [ ]:
merged_df.columns